<a href="https://colab.research.google.com/github/akhi-lbj/SQLGuard/blob/main/Ablation_Tests_Exp_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Create isolated full_dev directory inside bird_data
FULL_DEV_DIR = "/content/drive/MyDrive/SQLGuard_BIRD/bird_data/full_dev"
os.makedirs(FULL_DEV_DIR, exist_ok=True)
os.chdir(FULL_DEV_DIR)

print(f"Target directory set to: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Target directory set to: /content/drive/MyDrive/SQLGuard_BIRD/bird_data/full_dev


In [ ]:
ls

dev_20240627/


In [ ]:
cd dev_20240627/

/content/drive/MyDrive/SQLGuard_BIRD/bird_data/full_dev/dev_20240627


In [ ]:
import os
import re
import json
import time
import shutil
import sqlite3
import threading
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Optional, Dict, Any, TypedDict
from pydantic import BaseModel, Field
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import sqlglot
from sqlglot import parse_one, exp
from openai import OpenAI
from tqdm import tqdm
from langgraph.graph import StateGraph, START, END

# =====================================================================
# 0. DIRECTORY PATHS, FULL_DEV RESOLUTION & RESULTS CLEANING
# =====================================================================
DRIVE_SOURCE_DIR = "/content/drive/MyDrive/SQLGuard_BIRD/bird_data/full_dev/dev_20240627"
LOCAL_WORKING_DIR = "/content/sqlguard_run"
LOCAL_FULL_DEV_DIR = os.path.join(LOCAL_WORKING_DIR, "full_dev")
LOCAL_RESULTS_DIR = os.path.join(LOCAL_FULL_DEV_DIR, "exp_4_result")
LOCAL_RESULTS_FILE = os.path.join(LOCAL_RESULTS_DIR, "sqlguard_results_exp4.jsonl")
LOCAL_MEMORY_FILE = os.path.join(LOCAL_RESULTS_DIR, "sqlguard_failure_memory_exp4.json")

os.makedirs(LOCAL_WORKING_DIR, exist_ok=True)
file_lock = threading.Lock()
cache_lock = threading.Lock()
gpu_model_lock = threading.Lock()


def get_drive_results_dir() -> str:
    """Returns the target Drive results directory."""
    return os.path.join(DRIVE_SOURCE_DIR, "exp_4_result")


def clean_and_setup_results_dir():
    """Wipes any previous execution artifacts locally and on Drive."""
    # 1. Clean local NVMe results directory
    if os.path.exists(LOCAL_RESULTS_DIR):
        print(f">>> Removing previous local results in {LOCAL_RESULTS_DIR}...")
        shutil.rmtree(LOCAL_RESULTS_DIR)
    os.makedirs(LOCAL_RESULTS_DIR, exist_ok=True)

    # 2. Clean Google Drive results directory
    drive_results_dir = get_drive_results_dir()
    if os.path.exists(drive_results_dir):
        print(f">>> Removing previous Drive results in {drive_results_dir}...")
        shutil.rmtree(drive_results_dir)
    os.makedirs(drive_results_dir, exist_ok=True)

    print(">>> [READY] Fresh exp_4_result directory initialized.")


def setup_local_colab_environment():
    """Copies dataset files from Google Drive to local NVMe storage."""
    if os.path.exists(DRIVE_SOURCE_DIR) and not os.path.exists(LOCAL_FULL_DEV_DIR):
        print(f">>> Staging dataset from {DRIVE_SOURCE_DIR} to local NVMe ({LOCAL_FULL_DEV_DIR})...")
        shutil.copytree(DRIVE_SOURCE_DIR, LOCAL_FULL_DEV_DIR)
        print(">>> Staging complete!")
    elif os.path.exists(LOCAL_FULL_DEV_DIR):
        print(f">>> Local full_dev dataset ready at {LOCAL_FULL_DEV_DIR}.")


def sync_results_to_drive():
    """Atomically syncs incremental results and failure memory back to Drive."""
    try:
        drive_results_dir = get_drive_results_dir()
        os.makedirs(drive_results_dir, exist_ok=True)
        if os.path.exists(LOCAL_RESULTS_FILE):
            shutil.copy(LOCAL_RESULTS_FILE, os.path.join(drive_results_dir, "sqlguard_results_exp4.jsonl"))
        if os.path.exists(LOCAL_MEMORY_FILE):
            shutil.copy(LOCAL_MEMORY_FILE, os.path.join(drive_results_dir, "sqlguard_failure_memory_exp4.json"))
        print(f"\n>>> [SYNC SUCCESS] Checkpointed results to Drive: {drive_results_dir}")
    except Exception as e:
        print(f"\n>>> [SYNC ERROR] Drive backup failed: {e}")


# =====================================================================
# 1. K2-THINK-V2 ROTATING API MANAGER
# =====================================================================
K2_BASE_URL = os.getenv("K2_BASE_URL", "https://api.k2think.ai/v1")
MODEL_NAME = os.getenv("MODEL_NAME", "MBZUAI-IFM/K2-Think-v2")

API_KEYS = [
    "IFM-93mLmFQS2bfYZZIY",
    "IFM-SQJYZjtO76Kk86de",
    "IFM-BDZ81VHhqLyYn6Ka"
]


def clean_reasoning_output(raw_text: Optional[str]) -> str:
    """Strips <think> tags, markdown fences, and extracts clean JSON/SQL."""
    if not raw_text:
        return ""
    cleaned = re.sub(r"<think>.*?</think>", "", str(raw_text), flags=re.DOTALL).strip()

    if "```json" in cleaned:
        cleaned = cleaned.split("```json")[1].split("```")[0].strip()
    elif "```sql" in cleaned:
        cleaned = cleaned.split("```sql")[1].split("```")[0].strip()
    elif "```" in cleaned:
        cleaned = cleaned.split("```")[1].split("```")[0].strip()

    if not cleaned.startswith("{") and "select" in cleaned.lower():
        select_pos = cleaned.lower().find("select")
        if select_pos != -1:
            cleaned = cleaned[select_pos:].strip()
            cleaned = cleaned.replace("```", "").strip()

    return cleaned


class K2DynamicKeyManager:
    """Round-robin load balancer across active API keys with latency fallback."""
    def __init__(self, keys: List[str], base_url: str):
        self.keys = [k for k in keys if k and not k.startswith("YOUR_")]
        self.base_url = base_url
        self.clients = {k: OpenAI(base_url=self.base_url, api_key=k, timeout=60.0) for k in self.keys}
        self.index = 0
        self.lock = threading.Lock()

    def get_client_and_key(self) -> tuple[OpenAI, str]:
        with self.lock:
            key = self.keys[self.index % len(self.keys)]
            self.index += 1
            return self.clients[key], key

    def rotate_away_from(self, slow_key: str):
        with self.lock:
            if self.keys[self.index % len(self.keys)] == slow_key:
                self.index += 1

    def execute_chat_completion(self, prompt: str, max_retries: int = 2) -> str:
        for attempt in range(max_retries + 1):
            client, active_key = self.get_client_and_key()
            start_time = time.time()
            try:
                resp = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.0
                )
                if time.time() - start_time > 50.0:
                    self.rotate_away_from(active_key)
                content = resp.choices[0].message.content if resp.choices else ""
                return clean_reasoning_output(content)
            except Exception:
                self.rotate_away_from(active_key)
                if attempt == max_retries:
                    return ""
                time.sleep(1.0)
        return ""


llm_manager = K2DynamicKeyManager(API_KEYS, K2_BASE_URL)


# =====================================================================
# 2. LOCAL CODES-7B ENGINE (WITH EVIDENCE CHECKPOINT & SFT FORMAT)
# =====================================================================
class LocalCodeSEngine:
    """Thread-safe GPU inference manager for CodeS-7B with evidence conditioning."""
    def __init__(self, model_id: str = "seeklhy/codes-7b-bird-with-evidence"):
        print(f">>> Loading local SQL generator: {model_id}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        self.model.eval()
        print(">>> CodeS-7B (With Evidence) loaded successfully onto GPU.")

    def generate_sql(self, schema_str: str, question: str, evidence: str) -> str:
        """Constructs the canonical CodeS SFT prompt format to preserve native accuracy."""
        prompt = (
            f"Given the database schema, you need to translate the natural language question into SQL query.\n\n"
            f"[Database schema]\n{schema_str}\n\n"
            f"[Question]\n{question}\n\n"
            f"[Evidence]\n{evidence}\n\n"
            f"[SQL]\nSELECT"
        )
        with gpu_model_lock:
            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
            with torch.no_grad():
                output_ids = self.model.generate(
                    **inputs,
                    max_new_tokens=160,
                    pad_token_id=self.tokenizer.eos_token_id,
                    do_sample=False,
                    num_beams=4
                )
            generated = "SELECT" + self.tokenizer.decode(
                output_ids[0][inputs.input_ids.shape[1]:],
                skip_special_tokens=True
            )
        return clean_reasoning_output(generated)


codes_engine = LocalCodeSEngine()


# =====================================================================
# 3. THREAD-SAFE PERSISTENT FAILURE MEMORY
# =====================================================================
class PersistentFailureMemory:
    """Maintains an on-disk few-shot repository of repaired SQL patterns."""
    def __init__(self, memory_filepath: str = LOCAL_MEMORY_FILE):
        self.filepath = memory_filepath
        self.memory: List[Dict[str, Any]] = []
        self.reload()

    def reload(self):
        if os.path.exists(self.filepath):
            try:
                with open(self.filepath, "r") as f:
                    self.memory = json.load(f)
            except Exception:
                self.memory = []
        else:
            self.memory = []

    def record_repair(self, question: str, failed_sql: str, error_feedback: str, fixed_sql: str, error_types: List[str]):
        entry = {
            "question": question,
            "failed_sql": failed_sql,
            "error_feedback": error_feedback,
            "fixed_sql": fixed_sql,
            "error_types": error_types
        }
        with file_lock:
            self.memory.append(entry)
            with open(self.filepath, "w") as f:
                json.dump(self.memory, f, indent=2)

    def retrieve_similar_repairs(self, current_errors: List[str], max_examples: int = 2) -> str:
        with file_lock:
            if not self.memory:
                return ""
            mem_snapshot = list(self.memory)

        retrieved = []
        for entry in reversed(mem_snapshot):
            for err in current_errors:
                if any(err_type.lower() in err.lower() for err_type in entry.get("error_types", [])):
                    retrieved.append(entry)
                    break
            if len(retrieved) >= max_examples:
                break

        if not retrieved:
            retrieved = mem_snapshot[-max_examples:]

        formatted_cases = []
        for idx, item in enumerate(retrieved, 1):
            formatted_cases.append(
                f"[Past Repair Example #{idx}]\n"
                f"Question: {item['question']}\n"
                f"Failed Query: {item['failed_sql']}\n"
                f"Errors: {item['error_feedback']}\n"
                f"Corrected SQL: {item['fixed_sql']}"
            )
        return "\n\n".join(formatted_cases)


global_memory = PersistentFailureMemory()


# =====================================================================
# 4. 7-DIMENSIONAL SEMANTIC CONTRACT SCHEMA (Γ)
# =====================================================================
class TargetProjection(BaseModel):
    entity: str = Field(description="Summary of target projection")
    output_columns: List[str] = Field(default_factory=list, description="Projected expressions")
    granularity: Optional[str] = Field(default=None, description="Granularity level")

class SchemaLinks(BaseModel):
    required_tables: List[str] = Field(default_factory=list, description="Required tables")
    required_columns: List[str] = Field(default_factory=list, description="Required columns")
    join_keys: List[str] = Field(default_factory=list, description="Join paths")

class Analytics(BaseModel):
    aggregations: List[str] = Field(default_factory=list, description="COUNT, AVG, SUM, MIN, MAX")
    group_by: List[str] = Field(default_factory=list, description="Grouping columns or date slice expressions")
    having: List[str] = Field(default_factory=list, description="HAVING conditions")

class RankingCardinality(BaseModel):
    order_by: List[str] = Field(default_factory=list, description="Sort expressions")
    direction: Optional[str] = Field(default=None, description="ASC or DESC")
    limit: Optional[int] = Field(default=None, description="LIMIT top-k cap")

class SemanticContract(BaseModel):
    target_projection: TargetProjection
    schema_links: SchemaLinks
    predicates: List[str] = Field(default_factory=list, description="WHERE filters preserving INTEGER affinity")
    analytics: Analytics
    ranking_cardinality: RankingCardinality
    read_only: bool = Field(default=True, description="Strict read-only safety flag")
    ambiguity_flag: bool = Field(default=False, description="Ambiguity status")


# =====================================================================
# 5. COMPACT SCHEMA EXTRACTOR & CACHING
# =====================================================================
SCHEMA_CACHE: Dict[str, str] = {}


def extract_compact_schema(db_path: Optional[str]) -> str:
    """Builds a token-efficient, type-annotated SQLite schema description."""
    if not db_path or not os.path.exists(db_path):
        return "Schema unavailable."

    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type IN ('table', 'view') AND name NOT LIKE 'sqlite_%';")
        tables = [r[0] for r in cursor.fetchall()]

        schema_lines = []
        for table_name in tables:
            cursor.execute(f"PRAGMA table_info('{table_name}');")
            cols = cursor.fetchall()
            col_desc = [f"{c[1]} ({c[2].upper() or 'TEXT'})" for c in cols]
            schema_lines.append(f"TABLE {table_name} (\n  " + ", ".join(col_desc) + "\n)")

            cursor.execute(f"PRAGMA foreign_key_list('{table_name}');")
            for fk in cursor.fetchall():
                schema_lines.append(f"-- FK: {table_name}.{fk[3]} -> {fk[2]}.{fk[4]}")

            samples = []
            sampled_count = 0
            for col in cols:
                if sampled_count >= 4:
                    break
                col_name = col[1]
                cursor.execute(f"SELECT DISTINCT \"{col_name}\" FROM \"{table_name}\" WHERE \"{col_name}\" IS NOT NULL LIMIT 3;")
                vals = [r[0] for r in cursor.fetchall() if r[0] is not None]
                if vals:
                    samples.append(f"{col_name}: {vals}")
                    sampled_count += 1
            if samples:
                schema_lines.append(f"-- [{table_name} Samples]: " + " | ".join(samples))

        conn.close()
        return "\n".join(schema_lines)
    except Exception as e:
        return f"Error reading schema: {e}"


def get_cached_schema(db_id: str, db_path: Optional[str]) -> str:
    with cache_lock:
        if db_id in SCHEMA_CACHE:
            return SCHEMA_CACHE[db_id]

    compact_schema = extract_compact_schema(db_path)
    with cache_lock:
        SCHEMA_CACHE[db_id] = compact_schema
    return compact_schema


# =====================================================================
# 6. NORMALIZED HYBRID AST VALIDATOR (sqlglot)
# =====================================================================
class SQLGuardValidator:
    def validate(self, sql: str, contract: SemanticContract) -> Dict[str, Any]:
        errors = []
        error_types = []

        if not sql or not sql.strip():
            return {"passed": False, "errors": ["Generated SQL was empty."], "error_types": ["Syntax"]}

        try:
            parsed = parse_one(sql, read="sqlite")
        except Exception as e:
            return {"passed": False, "errors": [f"AST Parse Error: {str(e)}"], "error_types": ["Syntax"]}

        if not isinstance(parsed, exp.Select):
            return {
                "passed": False,
                "errors": ["Unit Test [V_safety] Failed: Non-SELECT operation blocked."],
                "error_types": ["Safety"]
            }

        query_tables = {t.name.lower().strip("`'\"[] ") for t in parsed.find_all(exp.Table)}
        query_columns_bare = {c.name.lower().strip("`'\"[] ") for c in parsed.find_all(exp.Column)}

        # 1. Table Verification
        for req_t in contract.schema_links.required_tables:
            req_t_clean = req_t.lower().strip("`'\"[] ")
            if req_t_clean and req_t_clean not in query_tables:
                errors.append(f"Unit Test [Schema Table] Failed: Required table '{req_t}' missing.")
                error_types.append("SchemaTable")

        # 2. Column Verification (Bare vs Qualified)
        for req_c in contract.schema_links.required_columns:
            req_clean = req_c.lower().strip("`'\"[] ")
            req_bare = req_clean.split(".")[-1].strip("`'\"[] ")
            if req_bare and req_bare not in query_columns_bare:
                errors.append(f"Unit Test [Schema Column] Failed: Required column '{req_c}' missing.")
                error_types.append("SchemaColumn")

        # 3. Aggregations
        if contract.analytics.aggregations:
            ast_funcs = set()
            if parsed.find(exp.Count): ast_funcs.add("count")
            if parsed.find(exp.Sum): ast_funcs.add("sum")
            if parsed.find(exp.Avg): ast_funcs.add("avg")
            if parsed.find(exp.Max): ast_funcs.add("max")
            if parsed.find(exp.Min): ast_funcs.add("min")

            for req_agg in contract.analytics.aggregations:
                req_clean = req_agg.lower().strip()
                matched = any(kw in req_clean and kw in ast_funcs for kw in ["count", "sum", "avg", "max", "min"])
                if not matched:
                    errors.append(f"Unit Test [Aggregation] Failed: Missing required function '{req_agg}'.")
                    error_types.append("Aggregation")

        # 4. Predicates
        if contract.predicates:
            has_where = parsed.find(exp.Where) is not None
            has_having = parsed.find(exp.Having) is not None
            if not (has_where or has_having):
                errors.append("Unit Test [Predicate] Failed: Filters specified in contract but WHERE/HAVING missing.")
                error_types.append("Predicate")

        # 5. Group By
        if contract.analytics.group_by and not parsed.find(exp.Group):
            errors.append("Unit Test [GroupBy] Failed: Contract specifies grouping but GROUP BY clause missing.")
            error_types.append("GroupBy")

        # 6. Order By & Limit
        if contract.ranking_cardinality.direction and not parsed.find(exp.Order):
            errors.append("Unit Test [Ranking] Failed: Contract specifies sort order but ORDER BY clause missing.")
            error_types.append("Ranking")

        if contract.ranking_cardinality.limit is not None and not parsed.find(exp.Limit):
            errors.append("Unit Test [Limit] Failed: Contract specifies top-k cap but LIMIT clause missing.")
            error_types.append("Limit")

        return {
            "passed": len(errors) == 0,
            "errors": errors,
            "error_types": list(set(error_types))
        }


# =====================================================================
# 7. LANGGRAPH WORKFLOW NODES
# =====================================================================
class SQLGuardState(TypedDict):
    question: str
    evidence: str
    db_id: str
    db_path: str
    gold_sql: str
    schema_metadata: str
    contract: Optional[SemanticContract]
    current_sql: str
    validation_passed: bool
    validation_errors: List[str]
    validation_error_types: List[str]
    attempt_count: int
    max_attempts: int
    initial_failed_sql: str
    initial_errors: List[str]
    initial_error_types: List[str]
    ex_passed: bool
    execution_result: Optional[List[Any]]
    audit_record: Dict[str, Any]


# EXP-4 ABLATION: No Table-Level Pruning (uses full extracted schema directly)
def schema_linker_node(state: SQLGuardState) -> Dict[str, Any]:
    schema_meta = extract_compact_schema(state["db_path"])
    return {"schema_metadata": schema_meta}


def intent_agent_node(state: SQLGuardState) -> Dict[str, Any]:
    prompt = f"""You are the Intent Agent for SQLGuard. Extract a 7-dimensional Semantic Contract as a JSON object.

### MANDATORY INSTRUCTIONS:
1. DOMAIN EVIDENCE GROUNDING: Strictly follow domain evidence. If evidence indicates date slicing (e.g. SUBSTR(Date, 5, 2) for month, SUBSTR(Date, 1, 4) for year), use that exact expression in output_columns and group_by.
2. SQLITE TYPE COMPLIANCE: If column type is INTEGER (e.g. Date 201301), do NOT wrap numeric numbers in single quotes (use `Date BETWEEN 201301 AND 201312`).
3. PROJECTION PRECISION: Output ONLY the requested attribute or expression in output_columns.

### BENCHMARK EVALUATION GROUNDING RULES:
1. NAME PROJECTION: When asked for a person's name or full name, project two separate columns `first_name, last_name` (or `forename, surname`). DO NOT concatenate with `|| ' ' ||`.
2. CASE-INSENSITIVE TEXT FILTERS: For string equality checks in WHERE clauses, use `COLLATE NOCASE` or `LIKE` (e.g. `Segment = 'Discount' COLLATE NOCASE`).
3. PROJECTION MINIMALISM: Project ONLY the exact attribute requested. Do not include extra tie-breaker columns or IDs in SELECT unless explicitly requested.
4. NULL-SAFE SUMS: Always provide `ELSE 0` in conditional aggregation (e.g., `SUM(CASE WHEN condition THEN val ELSE 0 END)`).

Schema:
{state["schema_metadata"]}

User Question: {state["question"]}
Domain Evidence: {state["evidence"]}

JSON Schema:
{json.dumps(SemanticContract.model_json_schema())}

Output ONLY the raw JSON object."""

    raw_json = llm_manager.execute_chat_completion(prompt)
    try:
        contract = SemanticContract.model_validate_json(raw_json)
    except Exception:
        contract = SemanticContract(
            target_projection=TargetProjection(entity=state["question"]),
            schema_links=SchemaLinks(),
            analytics=Analytics(),
            ranking_cardinality=RankingCardinality()
        )
    return {"contract": contract}


def generator_decomposer_node(state: SQLGuardState) -> Dict[str, Any]:
    generated_sql = codes_engine.generate_sql(
        schema_str=state["schema_metadata"],
        question=state["question"],
        evidence=state["evidence"]
    )
    return {"current_sql": generated_sql}


def hybrid_validator_node(state: SQLGuardState) -> Dict[str, Any]:
    validator = SQLGuardValidator()
    val = validator.validate(state["current_sql"], state["contract"])

    updates = {
        "validation_passed": val["passed"],
        "validation_errors": val["errors"],
        "validation_error_types": val.get("error_types", [])
    }

    if not val["passed"] and state["attempt_count"] == 0:
        updates["initial_failed_sql"] = state["current_sql"]
        updates["initial_errors"] = val["errors"]
        updates["initial_error_types"] = val.get("error_types", [])

    return updates


def repair_agent_node(state: SQLGuardState) -> Dict[str, Any]:
    memory_ctx = global_memory.retrieve_similar_repairs(state["validation_errors"])
    contract_json = state["contract"].model_dump_json() if state["contract"] else "{}"

    prompt = f"""Repair the following SQLite query to satisfy the Semantic Contract and Domain Evidence.

Schema:
{state["schema_metadata"]}

Question: {state["question"]}
Evidence: {state["evidence"]}
Semantic Contract: {contract_json}

[FAILED CANDIDATE SQL]:
{state["current_sql"]}

[CONTRACT VALIDATION ERRORS]:
{chr(10).join(state["validation_errors"])}
"""
    if memory_ctx:
        prompt += f"\n[SIMILAR PAST SUCCESSFUL REPAIRS]:\n{memory_ctx}\n"

    prompt += "\nOutput raw repaired SQL inside a ```sql codeblock."

    repaired_sql = llm_manager.execute_chat_completion(prompt)
    if not repaired_sql:
        repaired_sql = state["current_sql"]

    return {
        "current_sql": repaired_sql,
        "attempt_count": state["attempt_count"] + 1
    }


def execution_gate_node(state: SQLGuardState) -> Dict[str, Any]:
    ex_passed = False
    pred_res = None

    if state["validation_passed"] and state["db_path"] and os.path.exists(state["db_path"]):
        conn = sqlite3.connect(state["db_path"], timeout=10.0)
        cursor = conn.cursor()
        try:
            cursor.execute(state["current_sql"])
            pred_res = cursor.fetchall()

            cursor.execute(state["gold_sql"])
            gold_res = cursor.fetchall()

            # Robust float normalization comparison
            def normalize_cell(val):
                if isinstance(val, float):
                    return round(val, 2)
                return val

            def normalize_rows(rows):
                if not rows:
                    return []
                return [tuple(normalize_cell(c) for c in row) for row in rows]

            pred_norm = normalize_rows(pred_res)
            gold_norm = normalize_rows(gold_res)

            ex_passed = (pred_norm == gold_norm or set(pred_norm) == set(gold_norm))
        except Exception:
            ex_passed = False
        finally:
            conn.close()

    if state["validation_passed"] and state["attempt_count"] > 0 and state["initial_failed_sql"]:
        global_memory.record_repair(
            question=state["question"],
            failed_sql=state["initial_failed_sql"],
            error_feedback="\n".join(state["initial_errors"]),
            fixed_sql=state["current_sql"],
            error_types=state["initial_error_types"]
        )

    audit_record = {
        "question": state["question"],
        "db_id": state["db_id"],
        "contract": state["contract"].model_dump() if state["contract"] else {},
        "final_sql": state["current_sql"],
        "validation_passed": state["validation_passed"],
        "attempt_count": state["attempt_count"],
        "ex_passed": ex_passed
    }

    return {
        "ex_passed": ex_passed,
        "execution_result": pred_res,
        "audit_record": audit_record
    }


# =====================================================================
# 8. LANGGRAPH COMPILATION & LIVE BACKGROUND MONITOR
# =====================================================================
def validation_router(state: SQLGuardState) -> str:
    if state["validation_passed"]:
        return "execution_gate"
    if state["attempt_count"] < state["max_attempts"]:
        return "repair_agent"
    return "execution_gate"


workflow = StateGraph(SQLGuardState)

workflow.add_node("schema_linker", schema_linker_node)
workflow.add_node("intent_agent", intent_agent_node)
workflow.add_node("generator_decomposer", generator_decomposer_node)
workflow.add_node("hybrid_validator", hybrid_validator_node)
workflow.add_node("repair_agent", repair_agent_node)
workflow.add_node("execution_gate", execution_gate_node)

workflow.add_edge(START, "schema_linker")
workflow.add_edge("schema_linker", "intent_agent")
workflow.add_edge("intent_agent", "generator_decomposer")
workflow.add_edge("generator_decomposer", "hybrid_validator")

workflow.add_conditional_edges(
    "hybrid_validator",
    validation_router,
    {
        "execution_gate": "execution_gate",
        "repair_agent": "repair_agent"
    }
)

workflow.add_edge("repair_agent", "hybrid_validator")
workflow.add_edge("execution_gate", END)

sqlguard_app = workflow.compile()


def process_single_sample(sample: Dict[str, Any], db_map: Dict[str, str]) -> Dict[str, Any]:
    db_id = sample["db_id"]
    db_path = db_map.get(db_id, "")

    initial_state: SQLGuardState = {
        "question": sample["question"],
        "evidence": sample.get("evidence", ""),
        "db_id": db_id,
        "db_path": db_path,
        "gold_sql": sample["SQL"],
        "schema_metadata": "",
        "contract": None,
        "current_sql": "",
        "validation_passed": False,
        "validation_errors": [],
        "validation_error_types": [],
        "attempt_count": 0,
        "max_attempts": 3,
        "initial_failed_sql": "",
        "initial_errors": [],
        "initial_error_types": [],
        "ex_passed": False,
        "execution_result": None,
        "audit_record": {}
    }

    final_state = sqlguard_app.invoke(initial_state)

    with file_lock:
        with open(LOCAL_RESULTS_FILE, "a") as f_out:
            f_out.write(json.dumps(final_state["audit_record"]) + "\n")

    return final_state


def live_progress_logger(stop_event: threading.Event, total_target: int, poll_interval: float = 10.0):
    """Background thread that prints real-time accuracy and recovery metrics for EXP-4."""
    while not stop_event.is_set():
        if os.path.exists(LOCAL_RESULTS_FILE):
            records = []
            with file_lock:
                with open(LOCAL_RESULTS_FILE, "r") as f:
                    for line in f:
                        line = line.strip()
                        if line:
                            try:
                                records.append(json.loads(line))
                            except json.JSONDecodeError:
                                continue

            n = len(records)
            if n > 0:
                ex_pass = sum(1 for r in records if r.get("ex_passed", False))
                val_pass = sum(1 for r in records if r.get("validation_passed", False))
                repairs = [r for r in records if r.get("attempt_count", 0) > 0]
                rep_ex = sum(1 for r in repairs if r.get("ex_passed", False))

                pct_done = (n / total_target) * 100
                ex_acc = (ex_pass / n) * 100
                val_rate = (val_pass / n) * 100
                rep_acc = (rep_ex / len(repairs) * 100) if repairs else 0.0

                print(
                    f"\n[LIVE EXP-4 MONITOR] Evaluated: {n}/{total_target} ({pct_done:.1f}%) | "
                    f"EX Acc: {ex_acc:.2f}% | AST Valid: {val_rate:.1f}% | "
                    f"Repairs Recovered: {rep_ex}/{len(repairs)} ({rep_acc:.1f}%)"
                )

        stop_event.wait(poll_interval)


def run_full_bird_benchmark(
    limit_samples: Optional[int] = None,
    dataset_file: str = "dev.json",
    data_dir: str = LOCAL_FULL_DEV_DIR,
    max_workers: int = 6
):
    # 1. Setup local environment & clean previous results in exp_4_result
    setup_local_colab_environment()
    clean_and_setup_results_dir()
    global_memory.reload()

    # 2. Locate evaluation JSON
    json_path = None
    for root, _, files in os.walk(data_dir):
        if dataset_file in files:
            json_path = os.path.join(root, dataset_file)
            break

    if not json_path:
        raise FileNotFoundError(f"Could not locate {dataset_file} in '{data_dir}'.")

    with open(json_path, "r") as f:
        full_data = json.load(f)
        samples = full_data[:limit_samples] if limit_samples is not None else full_data

    # 3. Map SQLite databases
    db_map = {}
    for root, _, files in os.walk(data_dir):
        for file in files:
            if file.endswith(".sqlite"):
                db_id = file.replace(".sqlite", "")
                db_map[db_id] = os.path.join(root, file)

    print(f">>> Found {len(db_map)} SQLite databases in {data_dir}.")
    print(f">>> Total dev set samples to evaluate: {len(samples)}")

    passed_semantic_gate = 0
    correct_execution_count = 0
    total_repaired_count = 0

    print(f"\n=================== RUNNING HYBRID SQLGUARD EXP-4 NO PRUNING ({len(samples)} SAMPLES) ===================")

    # 4. Start background monitor thread
    stop_monitor_event = threading.Event()
    monitor_thread = threading.Thread(
        target=live_progress_logger,
        args=(stop_monitor_event, len(samples), 15.0),
        daemon=True
    )
    monitor_thread.start()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_single_sample, sample, db_map) for sample in samples]

        for idx, future in enumerate(tqdm(as_completed(futures), total=len(samples)), 1):
            try:
                final_state = future.result()
                if final_state["validation_passed"]:
                    passed_semantic_gate += 1
                    if final_state["attempt_count"] > 0:
                        total_repaired_count += 1
                if final_state["ex_passed"]:
                    correct_execution_count += 1
            except Exception as e:
                print(f"Sample processing error: {e}")

            if idx % 25 == 0:
                sync_results_to_drive()

    # 5. Stop monitor & perform final sync
    stop_monitor_event.set()
    monitor_thread.join(timeout=1.0)
    sync_results_to_drive()

    total = len(samples)
    print("\n=================== FULL BIRD FINAL BENCHMARK METRICS (EXP-4) ===================")
    print(f"Total Samples Evaluated        : {total}")
    print(f"Passed Semantic Contract Gate  : {passed_semantic_gate}/{total} ({passed_semantic_gate/total*100:.1f}%)")
    print(f"Successfully Repaired Queries  : {total_repaired_count}")
    print(f"BIRD Execution Accuracy (EX)   : {correct_execution_count}/{total} ({correct_execution_count/total*100:.1f}%)")
    print(f"Local Results Directory        : {LOCAL_RESULTS_DIR}")
    print(f"Google Drive Results Directory : {get_drive_results_dir()}")


if __name__ == "__main__":
    try:
        run_full_bird_benchmark(
            limit_samples=None,  # Evaluates all 1,534 samples
            dataset_file="dev.json",
            data_dir=LOCAL_FULL_DEV_DIR,
            max_workers=6
        )
    finally:
        # 1. Ensure any remaining files are synced to Google Drive
        sync_results_to_drive()

        # 2. Disconnect and release the Colab runtime automatically
        print("\n>>> [COMPLETE] Benchmark finished. Shutting down Colab runtime to save compute units...")
        time.sleep(5)  # Short buffer to ensure disk flush

        from google.colab import runtime
        runtime.unassign()

>>> Loading local SQL generator: seeklhy/codes-7b-bird-with-evidence...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

>>> CodeS-7B (With Evidence) loaded successfully onto GPU.
>>> Local full_dev dataset ready at /content/sqlguard_run/full_dev.
>>> Removing previous local results in /content/sqlguard_run/full_dev/exp_4_result...
>>> Removing previous Drive results in /content/drive/MyDrive/SQLGuard_BIRD/bird_data/full_dev/dev_20240627/exp_4_result...
>>> [READY] Fresh exp_4_result directory initialized.
>>> Found 11 SQLite databases in /content/sqlguard_run/full_dev.
>>> Total dev set samples to evaluate: 1534

=================== RUNNING HYBRID SQLGUARD EXP-4 NO PRUNING (1534 SAMPLES) ===================


  0%|          | 1/1534 [00:10<4:27:04, 10.45s/it]

...


[LIVE EXP-4 MONITOR] Evaluated: 1528/1534 (99.6%) | EX Acc: 56.35% | AST Valid: 98.8% | Repairs Recovered: 369/767 (48.1%)


100%|█████████▉| 1531/1534 [2:11:43<00:16,  5.61s/it]


[LIVE EXP-4 MONITOR] Evaluated: 1531/1534 (99.8%) | EX Acc: 56.24% | AST Valid: 98.8% | Repairs Recovered: 369/770 (47.9%)


100%|██████████| 1534/1534 [2:11:53<00:00,  5.16s/it]



>>> [SYNC SUCCESS] Checkpointed results to Drive: /content/drive/MyDrive/SQLGuard_BIRD/bird_data/full_dev/dev_20240627/exp_4_result

=================== FULL BIRD FINAL BENCHMARK METRICS (EXP-4) ===================
Total Samples Evaluated        : 1534
Passed Semantic Contract Gate  : 1515/1534 (98.8%)
Successfully Repaired Queries  : 752
BIRD Execution Accuracy (EX)   : 862/1534 (56.2%)
Local Results Directory        : /content/sqlguard_run/full_dev/exp_4_result
Google Drive Results Directory : /content/drive/MyDrive/SQLGuard_BIRD/bird_data/full_dev/dev_20240627/exp_4_result

>>> [SYNC SUCCESS] Checkpointed results to Drive: /content/drive/MyDrive/SQLGuard_BIRD/bird_data/full_dev/dev_20240627/exp_4_result

>>> [COMPLETE] Benchmark finished. Shutting down Colab runtime to save compute units...
